# Feature Engineering v1: Astram Event Data

Creates prediction-time-safe feature tables for road-closure and duration-band modelling. This version keeps targets, metadata, QA fields, and model features separate; removes known post-event leakage; builds strict past-only historical features; and defers imputation, rare-category fitting, and encoding until after a chronological training split.

## 1. Prediction-time contract

The feature snapshot is assumed to be created when an event is first reported. `created_date` is used as the prediction timestamp, with `start_datetime` as a fallback when creation time is missing. Event description, start location, and event context are treated as available then. Endpoints, final status, closure/resolution timestamps, resolver information, and duration outcomes are unavailable and cannot be model features.

In [74]:
from pathlib import Path

import numpy as np
import pandas as pd

HISTORY_SMOOTHING_ALPHA = 20.0
MIN_DURATION_GROUP_HISTORY = 5
DURATION_BOUNDARY_WINDOW_MINUTES = 5.0
BOUNDARY_MIN_EXACT_ROWS_WARN = 10
BOUNDARY_EXACT_SHARE_WARN = 0.50
BOUNDARY_ALIGNED_SHARE_WARN = 0.90
HIGH_MISSING_PERCENT_WARN = 80.0
DURATION_CLASSES = ['short', 'medium', 'long', 'very_long']
MISSING_HISTORY_KEY = '__missing__'
GLOBAL_HISTORY_KEY = '__all__'

## 2. Load and standardize the dataset

In [75]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data').exists()), cwd)
DATA_PATH = PROJECT_ROOT / 'data' / 'Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'feature_engineering_v1'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
df = raw_df.copy()
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)
df['_source_row'] = np.arange(len(df), dtype=int)

print('Raw shape:', raw_df.shape)
print('Standardized columns:', len(df.columns))

Raw shape: (8173, 46)
Standardized columns: 47


## 3. Parse timestamps and establish prediction order

In [76]:
datetime_cols = [
    'start_datetime',
    'end_datetime',
    'created_date',
    'modified_datetime',
    'closed_datetime',
    'resolved_datetime',
]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

if 'start_datetime' not in df.columns:
    raise ValueError('start_datetime is required for target and time features')

created_time = df['created_date'] if 'created_date' in df.columns else pd.Series(pd.NaT, index=df.index)
df['prediction_datetime'] = created_time.fillna(df['start_datetime'])
df['start_datetime_ist'] = df['start_datetime'].dt.tz_convert('Asia/Kolkata')
df['prediction_datetime_ist'] = df['prediction_datetime'].dt.tz_convert('Asia/Kolkata')

df = df.sort_values(
    ['prediction_datetime', 'start_datetime', '_source_row'],
    kind='mergesort',
    na_position='last',
).reset_index(drop=True)

print('Missing prediction timestamps:', int(df['prediction_datetime'].isna().sum()))

Missing prediction timestamps: 2


## 4. Clean prediction-time categories

In [77]:
category_cols = [
    'event_type', 'event_cause', 'direction', 'veh_type', 'corridor',
    'cargo_material', 'reason_breakdown', 'police_station', 'zone', 'junction',
]
null_tokens = {'': pd.NA, 'nan': pd.NA, 'none': pd.NA, 'null': pd.NA, '<na>': pd.NA, 'na': pd.NA}

for col in category_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype('string')
            .str.strip()
            .str.lower()
            .replace(null_tokens)
        )

if 'event_cause' in df.columns:
    df['event_cause'] = df['event_cause'].replace({
        'fog / low visibility': 'fog_low_visibility',
        'traffic congestion': 'congestion',
        'vehicle break down': 'vehicle_breakdown',
        'vehicle breakdown': 'vehicle_breakdown',
    })

# Cleaned for QA, but not selected until prediction-time availability is confirmed.
for source_col, clean_col in [('priority', 'priority_clean'), ('authenticated', 'authenticated_clean')]:
    if source_col in df.columns:
        df[clean_col] = (
            df[source_col].astype('string').str.strip().str.lower().replace(null_tokens)
        )

## 5. Build targets and duration QA

Road-closure values that cannot be parsed remain missing rather than being converted to no-closure. Duration uses the requested hierarchy: closed time, otherwise end time, otherwise resolved time. Negative alternatives are invalidated before the hierarchy is applied. Durations above 24 hours are labelled `very_long`. Alternative durations remain QA-only columns.

In [78]:
def parse_bool_target(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype('Int64')

    normalized = series.astype('string').str.strip().str.lower()
    parsed = pd.Series(pd.NA, index=series.index, dtype='Int64')
    parsed.loc[normalized.isin({'true', '1', 'yes', 'y'})] = 1
    parsed.loc[normalized.isin({'false', '0', 'no', 'n'})] = 0
    return parsed

if 'requires_road_closure' in df.columns:
    df['target_road_closure'] = parse_bool_target(df['requires_road_closure'])
    df['target_road_closure_missing'] = df['target_road_closure'].isna().astype('int')
else:
    df['target_road_closure'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
    df['target_road_closure_missing'] = 1

duration_hierarchy = [
    ('closed', 'closed_datetime'),
    ('end', 'end_datetime'),
    ('resolved', 'resolved_datetime'),
]
valid_duration_by_source = {}
negative_duration_rows = []
negative_flags = []

for source_name, timestamp_col in duration_hierarchy:
    qa_minutes_col = f'qa_duration_minutes_{source_name}'
    qa_negative_col = f'qa_negative_duration_{source_name}'

    if timestamp_col in df.columns:
        raw_minutes = (df[timestamp_col] - df['start_datetime']).dt.total_seconds() / 60
    else:
        raw_minutes = pd.Series(np.nan, index=df.index, dtype=float)

    negative_mask = raw_minutes.lt(0).fillna(False)
    df[qa_minutes_col] = raw_minutes
    df[qa_negative_col] = negative_mask.astype('int')
    valid_duration_by_source[source_name] = raw_minutes.mask(negative_mask)
    negative_flags.append(qa_negative_col)
    negative_duration_rows.append({
        'duration_source': source_name,
        'negative_duration_count': int(negative_mask.sum()),
    })

selected_duration = pd.Series(np.nan, index=df.index, dtype=float)
selected_source = pd.Series(pd.NA, index=df.index, dtype='string')
selected_available_at = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns, UTC]')

for source_name, timestamp_col in duration_hierarchy:
    candidate = valid_duration_by_source[source_name]
    choose = selected_duration.isna() & candidate.notna()
    selected_duration.loc[choose] = candidate.loc[choose]
    selected_source.loc[choose] = source_name
    if timestamp_col in df.columns:
        selected_available_at.loc[choose] = df.loc[choose, timestamp_col]

df['duration_minutes_raw'] = selected_duration
df['duration_source'] = selected_source
df['duration_label_available_at'] = selected_available_at
df['has_duration_label'] = selected_duration.notna().astype('int')
df['duration_band'] = pd.cut(
    df['duration_minutes_raw'],
    bins=[0, 60, 240, 24 * 60, np.inf],
    labels=DURATION_CLASSES,
    include_lowest=True,
)

valid_alternative_frame = pd.DataFrame(valid_duration_by_source)
df['qa_duration_valid_source_count'] = valid_alternative_frame.notna().sum(axis=1)
df['qa_duration_source_spread_minutes'] = (
    valid_alternative_frame.max(axis=1) - valid_alternative_frame.min(axis=1)
).where(df['qa_duration_valid_source_count'].ge(2))
df['qa_any_negative_duration'] = df[negative_flags].any(axis=1).astype('int')

negative_duration_rows.append({
    'duration_source': 'unique_rows_with_any_negative_duration',
    'negative_duration_count': int(df['qa_any_negative_duration'].sum()),
})
negative_duration_summary = pd.DataFrame(negative_duration_rows)

print('Missing road-closure targets:', int(df['target_road_closure'].isna().sum()))
print('Rows with a valid duration target:', int(df['has_duration_label'].sum()))
print(negative_duration_summary.to_string(index=False))
print(df['duration_band'].value_counts(dropna=False).sort_index())

Missing road-closure targets: 0
Rows with a valid duration target: 3498
                       duration_source  negative_duration_count
                                closed                        3
                                   end                       48
                              resolved                        0
unique_rows_with_any_negative_duration                       51
duration_band
short        1579
medium        903
long          280
very_long     736
NaN          4675
Name: count, dtype: int64


## 6. Inspect duration boundaries

Creates a row-level QA table around 60 and 240 minutes. Exact-boundary concentration and minute-aligned timestamps are reported so rounded or default timestamps can be investigated before modelling.

In [79]:
boundary_frames = []
for boundary in [60.0, 240.0]:
    distance = (df['duration_minutes_raw'] - boundary).abs()
    near_mask = distance.le(DURATION_BOUNDARY_WINDOW_MINUTES) & df['duration_minutes_raw'].notna()
    if not near_mask.any():
        continue

    boundary_part = df.loc[near_mask, [
        '_source_row', 'id', 'start_datetime', 'duration_label_available_at',
        'duration_minutes_raw', 'duration_source',
    ]].copy()
    boundary_part['boundary_minutes'] = boundary
    boundary_part['distance_from_boundary_minutes'] = distance.loc[near_mask]
    boundary_part['is_exact_boundary'] = np.isclose(
        boundary_part['duration_minutes_raw'], boundary, atol=1e-9
    )
    boundary_part['start_second_is_zero'] = boundary_part['start_datetime'].dt.second.eq(0)
    boundary_part['outcome_second_is_zero'] = boundary_part['duration_label_available_at'].dt.second.eq(0)
    boundary_part['both_timestamps_minute_aligned'] = (
        boundary_part['start_second_is_zero'] & boundary_part['outcome_second_is_zero']
    )
    boundary_part['exact_and_minute_aligned'] = (
        boundary_part['is_exact_boundary']
        & boundary_part['both_timestamps_minute_aligned']
    )
    boundary_frames.append(boundary_part)

duration_boundary_qa = (
    pd.concat(boundary_frames, ignore_index=True)
    if boundary_frames
    else pd.DataFrame(columns=[
        '_source_row', 'id', 'start_datetime', 'duration_label_available_at',
        'duration_minutes_raw', 'duration_source', 'boundary_minutes',
        'distance_from_boundary_minutes', 'is_exact_boundary',
        'start_second_is_zero', 'outcome_second_is_zero',
        'both_timestamps_minute_aligned', 'exact_and_minute_aligned',
    ])
)

if duration_boundary_qa.empty:
    duration_boundary_summary = pd.DataFrame(columns=[
        'boundary_minutes', 'duration_source', 'near_boundary_rows',
        'exact_boundary_rows', 'minute_aligned_rows',
        'exact_and_minute_aligned_rows', 'exact_share',
        'exact_aligned_share', 'status', 'action',
    ])
else:
    duration_boundary_summary = (
        duration_boundary_qa
        .groupby(['boundary_minutes', 'duration_source'], dropna=False)
        .agg(
            near_boundary_rows=('duration_minutes_raw', 'size'),
            exact_boundary_rows=('is_exact_boundary', 'sum'),
            minute_aligned_rows=('both_timestamps_minute_aligned', 'sum'),
            exact_and_minute_aligned_rows=('exact_and_minute_aligned', 'sum'),
        )
        .reset_index()
    )
    duration_boundary_summary['exact_share'] = (
        duration_boundary_summary['exact_boundary_rows']
        / duration_boundary_summary['near_boundary_rows']
    )
    duration_boundary_summary['exact_aligned_share'] = (
        duration_boundary_summary['exact_and_minute_aligned_rows']
        / duration_boundary_summary['exact_boundary_rows'].replace(0, np.nan)
    ).fillna(0)
    suspicious_boundary = (
        duration_boundary_summary['exact_boundary_rows'].ge(BOUNDARY_MIN_EXACT_ROWS_WARN)
        & duration_boundary_summary['exact_share'].ge(BOUNDARY_EXACT_SHARE_WARN)
        & duration_boundary_summary['exact_aligned_share'].ge(BOUNDARY_ALIGNED_SHARE_WARN)
    )
    duration_boundary_summary['status'] = np.where(suspicious_boundary, 'WARN', 'PASS')
    duration_boundary_summary['action'] = np.where(
        suspicious_boundary,
        'Possible rounded/default timestamps; review only the flagged boundary rows.',
        'No strong rounded-timestamp concentration detected.',
    )

duration_boundary_summary

,boundary_minutes,duration_source,near_boundary_rows,exact_boundary_rows,minute_aligned_rows,exact_and_minute_aligned_rows,exact_share,exact_aligned_share,status,action
0,60.0,closed,165,0,0,0,0.0,0.0,PASS,No strong rounded-timestamp concentration dete...
1,60.0,end,5,0,0,0,0.0,0.0,PASS,No strong rounded-timestamp concentration dete...
2,60.0,resolved,7,0,0,0,0.0,0.0,PASS,No strong rounded-timestamp concentration dete...
3,240.0,closed,5,0,0,0,0.0,0.0,PASS,No strong rounded-timestamp concentration dete...
4,240.0,end,7,0,0,0,0.0,0.0,PASS,No strong rounded-timestamp concentration dete...


## 7. Create prediction-time base features

In [80]:
start_col = 'start_datetime_ist'
if start_col in df.columns:
    df['start_hour'] = df[start_col].dt.hour
    df['start_dayofweek'] = df[start_col].dt.dayofweek
    df['start_day_name'] = df[start_col].dt.day_name().str.lower().astype('string')
    df['start_month_number'] = df[start_col].dt.month
    df['start_month_name'] = df[start_col].dt.month_name().str.lower().astype('string')
    df['start_weekofyear'] = df[start_col].dt.isocalendar().week.astype('Int64')
    df['is_weekend'] = df['start_dayofweek'].isin([5, 6]).astype('int')
    df['is_morning_peak'] = df['start_hour'].between(8, 11).astype('int')
    df['is_evening_peak'] = df['start_hour'].between(17, 20).astype('int')
    df['is_peak_hour'] = (df['is_morning_peak'].eq(1) | df['is_evening_peak'].eq(1)).astype('int')
    df['is_night'] = (df['start_hour'].ge(22) | df['start_hour'].le(5)).astype('int')
    df['hour_sin'] = np.sin(2 * np.pi * df['start_hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['start_hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['start_dayofweek'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['start_dayofweek'] / 7)
    df['peak_period'] = np.select(
        [df['is_morning_peak'].eq(1), df['is_evening_peak'].eq(1), df['is_night'].eq(1)],
        ['morning_peak', 'evening_peak', 'night'],
        default='off_peak',
    )

if {'created_date', 'start_datetime'}.issubset(df.columns):
    report_lag = (df['created_date'] - df['start_datetime']).dt.total_seconds() / 60
    df['report_lag_missing'] = report_lag.isna().astype('int')
    df['report_lag_is_negative'] = report_lag.lt(0).fillna(False).astype('int')
    df['report_lag_minutes_clipped'] = report_lag.clip(lower=0, upper=24 * 60)
    df['report_lag_hours_clipped'] = df['report_lag_minutes_clipped'] / 60

for col in ['latitude', 'longitude']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

if {'latitude', 'longitude'}.issubset(df.columns):
    df['valid_start_coordinate'] = (
        df['latitude'].between(12.0, 14.2) & df['longitude'].between(76.5, 78.5)
    ).astype('int')
    df['has_start_location'] = df[['latitude', 'longitude']].notna().all(axis=1).astype('int')
    df['lat_round_3'] = df['latitude'].where(df['valid_start_coordinate'].eq(1)).round(3)
    df['lon_round_3'] = df['longitude'].where(df['valid_start_coordinate'].eq(1)).round(3)
    df['location_grid'] = (
        df['lat_round_3'].astype('string') + '_' + df['lon_round_3'].astype('string')
    )
    df['distance_to_city_center_km'] = haversine_km(
        df['latitude'], df['longitude'], 12.9716, 77.5946
    )
    df.loc[df['valid_start_coordinate'].eq(0), 'distance_to_city_center_km'] = np.nan

In [81]:
# End address and comments are intentionally excluded because they may be populated after the event.
text_sources = [col for col in ['description', 'address'] if col in df.columns]
if text_sources:
    combined_text = df[text_sources].fillna('').astype(str).agg(' '.join, axis=1).str.lower()
    description_text = df.get('description', pd.Series('', index=df.index)).fillna('').astype(str)

    df['description_missing'] = description_text.str.strip().eq('').astype('int')
    df['text_length'] = combined_text.str.len()
    df['description_char_length'] = description_text.str.len()
    df['description_word_count'] = description_text.str.split().str.len()
    df['has_non_ascii_text'] = description_text.str.contains(r'[^\x00-\x7F]', regex=True).astype('int')
    df['has_kannada_text'] = description_text.str.contains(r'[\u0C80-\u0CFF]', regex=True).astype('int')
    keyword_patterns = {
        'has_accident_word': r'accident|crash|collision',
        'has_breakdown_word': r'breakdown|vehicle breakdown|stalled|puncture',
        'has_water_word': r'water|flood|logging|rain',
        'has_construction_word': r'construction|work|repair|cement',
        'has_event_word': r'event|rally|procession|festival|protest',
        'has_blocked_word': r'block|blocked|closure|closed|obstruction|barricade',
        'has_jam_word': r'jam|congestion|traffic|slow|pile',
        'has_vip_word': r'vip|minister|convoy|movement',
        'has_location_hint_word': r'junction|circle|road|main|cross|signal|bridge|gate|layout',
        'has_emergency_word': r'emergency|ambulance|fire|injury|injured|death',
    }
    for feature_name, pattern in keyword_patterns.items():
        df[feature_name] = combined_text.str.contains(pattern, regex=True).astype('int')

if 'event_type' in df.columns:
    df['is_planned_event'] = df['event_type'].eq('planned').astype('int')

if 'event_cause' in df.columns:
    df['is_public_or_vip_event'] = df['event_cause'].isin(
        ['public_event', 'procession', 'vip_movement', 'protest']
    ).astype('int')
    df['is_breakdown_event'] = df['event_cause'].eq('vehicle_breakdown').astype('int')
    df['is_accident_event'] = df['event_cause'].eq('accident').astype('int')
    df['is_weather_or_visibility_event'] = df['event_cause'].isin(
        ['water_logging', 'fog_low_visibility']
    ).astype('int')
    df['is_road_condition_event'] = df['event_cause'].isin(
        ['pot_holes', 'road_conditions', 'debris', 'construction']
    ).astype('int')

if 'veh_type' in df.columns:
    veh_text = df['veh_type'].fillna('').astype(str)
    df['has_vehicle_type'] = df['veh_type'].notna().astype('int')
    df['is_truck'] = veh_text.str.contains(r'truck|lorry|hgv|heavy', regex=True).astype('int')
    df['is_bus'] = veh_text.str.contains('bus', regex=False).astype('int')
    df['is_two_wheeler'] = veh_text.str.contains(r'bike|two|2|scooter|motor', regex=True).astype('int')
    df['is_heavy_vehicle'] = (df['is_truck'].eq(1) | df['is_bus'].eq(1)).astype('int')

if 'cargo_material' in df.columns:
    df['has_cargo_material'] = df['cargo_material'].notna().astype('int')

if 'age_of_truck' in df.columns:
    df['age_of_truck'] = pd.to_numeric(df['age_of_truck'], errors='coerce')
    df['truck_age_missing'] = df['age_of_truck'].isna().astype('int')
    df['truck_age_band'] = pd.cut(
        df['age_of_truck'], bins=[-1, 5, 10, 20, np.inf],
        labels=['new', 'mid', 'old', 'very_old'],
    ).astype('string')

In [82]:
def combine_categories(*series):
    result = series[0].astype('string').fillna('unknown')
    for value in series[1:]:
        result = result + '_' + value.astype('string').fillna('unknown')
    return result

if {'event_cause', 'corridor'}.issubset(df.columns):
    df['event_cause_corridor'] = combine_categories(df['event_cause'], df['corridor'])
if {'event_cause', 'peak_period'}.issubset(df.columns):
    df['cause_peak_interaction'] = combine_categories(df['event_cause'], df['peak_period'])
if {'event_cause', 'zone'}.issubset(df.columns):
    df['zone_cause_interaction'] = combine_categories(df['zone'], df['event_cause'])
if {'event_cause', 'corridor'}.issubset(df.columns):
    df['corridor_cause_interaction'] = combine_categories(df['corridor'], df['event_cause'])
if {'event_cause', 'is_heavy_vehicle'}.issubset(df.columns):
    df['cause_heavy_vehicle_interaction'] = combine_categories(
        df['event_cause'], 'heavy_' + df['is_heavy_vehicle'].astype('string')
    )
if {'corridor', 'peak_period'}.issubset(df.columns):
    df['corridor_peak_interaction'] = combine_categories(df['corridor'], df['peak_period'])
if {'event_cause', 'start_hour'}.issubset(df.columns):
    df['cause_hour_interaction'] = combine_categories(df['event_cause'], df['start_hour'])
if {'event_cause', 'veh_type'}.issubset(df.columns):
    df['vehicle_cause_interaction'] = combine_categories(df['veh_type'], df['event_cause'])
if {'cargo_material', 'veh_type'}.issubset(df.columns):
    df['cargo_vehicle_interaction'] = combine_categories(df['cargo_material'], df['veh_type'])
if {'event_type', 'peak_period'}.issubset(df.columns):
    df['planned_peak_interaction'] = combine_categories(df['event_type'], df['peak_period'])
if {'is_weather_or_visibility_event', 'zone'}.issubset(df.columns):
    df['weather_zone_interaction'] = combine_categories(
        'weather_' + df['is_weather_or_visibility_event'].astype('string'), df['zone']
    )

## 8. Build strict past-only historical features

All lookups use `allow_exact_matches=False`, so the current row and every event at the same prediction timestamp are excluded. Closure history uses earlier reported closure decisions. Duration history uses only labels whose selected closed/end/resolved timestamp is strictly earlier than the current prediction timestamp. No full-dataset target prior is used.

In [83]:
def history_key(series):
    return series.astype('string').fillna(MISSING_HISTORY_KEY)

def normalize_asof_time(series):
    # pandas requires identical datetime units on both sides of merge_asof.
    # CSV parsing can produce datetime64[us, UTC], while constructed label
    # timestamps may be datetime64[ns, UTC], so normalize both explicitly.
    return pd.to_datetime(series, errors='coerce', utc=True).astype('datetime64[ns, UTC]')

def strict_asof_lookup(data, query_time_col, query_keys, snapshots, snapshot_time_col, stat_cols):
    result = pd.DataFrame(np.nan, index=data.index, columns=stat_cols, dtype=float)
    if snapshots.empty:
        return result

    # A single vectorized merge is much faster than one merge per category.
    query = data[[query_time_col]].copy()
    query['_row_index'] = data.index
    query['_history_key'] = history_key(query_keys).astype(str).to_numpy()
    query['_asof_join_time'] = normalize_asof_time(query[query_time_col])
    query = query.dropna(subset=['_asof_join_time']).sort_values(
        ['_asof_join_time', '_row_index'], kind='mergesort'
    )

    history = snapshots[[snapshot_time_col, '_history_key', *stat_cols]].copy()
    history['_history_key'] = history_key(history['_history_key']).astype(str)
    history['_asof_join_time'] = normalize_asof_time(history[snapshot_time_col])
    history = history.dropna(subset=['_asof_join_time']).sort_values(
        '_asof_join_time', kind='mergesort'
    )
    if query.empty or history.empty:
        return result

    matched = pd.merge_asof(
        query, history[['_history_key', '_asof_join_time', *stat_cols]],
        on='_asof_join_time', by='_history_key',
        direction='backward', allow_exact_matches=False,
    )
    result.loc[matched['_row_index'].to_numpy(), stat_cols] = matched[stat_cols].to_numpy()

    return result

def make_event_count_snapshots(data, key_values, availability_col):
    work = data.loc[data[availability_col].notna(), [availability_col, '_source_row']].copy()
    work['_history_key'] = key_values.loc[work.index].to_numpy()
    work = work.sort_values(['_history_key', availability_col, '_source_row'], kind='mergesort')
    work['_snapshot_event_count'] = work.groupby('_history_key', sort=False).cumcount() + 1
    return work.groupby(['_history_key', availability_col], sort=False, dropna=False).tail(1)

def make_binary_snapshots(data, key_values, target_col, availability_col):
    valid = data[target_col].notna() & data[availability_col].notna()
    work = data.loc[valid, [target_col, availability_col, '_source_row']].copy()
    work[target_col] = work[target_col].astype(float)
    work['_history_key'] = key_values.loc[work.index].to_numpy()
    work = work.sort_values(['_history_key', availability_col, '_source_row'], kind='mergesort')
    grouped = work.groupby('_history_key', sort=False)
    work['_snapshot_binary_count'] = grouped.cumcount() + 1
    work['_snapshot_binary_sum'] = grouped[target_col].cumsum()
    return work.groupby(['_history_key', availability_col], sort=False, dropna=False).tail(1)

def make_duration_snapshots(data, key_values, availability_col):
    valid = (
        data['duration_minutes_raw'].notna()
        & data['duration_band'].notna()
        & data[availability_col].notna()
    )
    work = data.loc[valid, [
        'duration_minutes_raw', 'duration_band', availability_col, '_source_row',
    ]].copy()
    work['_history_key'] = key_values.loc[work.index].to_numpy()
    work = work.sort_values(['_history_key', availability_col, '_source_row'], kind='mergesort')
    grouped = work.groupby('_history_key', sort=False)
    expanding_duration = grouped['duration_minutes_raw'].expanding()
    work['_snapshot_duration_count'] = grouped.cumcount() + 1
    work['_snapshot_duration_q25'] = expanding_duration.quantile(0.25).reset_index(level=0, drop=True)
    work['_snapshot_duration_median'] = expanding_duration.median().reset_index(level=0, drop=True)
    work['_snapshot_duration_q75'] = expanding_duration.quantile(0.75).reset_index(level=0, drop=True)
    for duration_class in DURATION_CLASSES:
        class_hits = work['duration_band'].astype('string').eq(duration_class).astype(int)
        work[f'_snapshot_duration_rate_{duration_class}'] = (
            class_hits.groupby(work['_history_key'], sort=False).cumsum()
            / work['_snapshot_duration_count']
        )
    return work.groupby(['_history_key', availability_col], sort=False, dropna=False).tail(1)

def build_strict_past_event_count(data, group_col):
    keys = history_key(data[group_col])
    snapshots = make_event_count_snapshots(data, keys, 'prediction_datetime')
    looked_up = strict_asof_lookup(
        data, 'prediction_datetime', keys, snapshots, 'prediction_datetime',
        ['_snapshot_event_count'],
    )
    return pd.DataFrame({
        f'past_count_{group_col}': looked_up['_snapshot_event_count'].fillna(0).astype(int),
    }, index=data.index)

def build_past_closure_features(data, group_col, global_rate):
    keys = history_key(data[group_col])
    snapshots = make_binary_snapshots(
        data, keys, 'target_road_closure', 'prediction_datetime'
    )
    looked_up = strict_asof_lookup(
        data, 'prediction_datetime', keys, snapshots, 'prediction_datetime',
        ['_snapshot_binary_count', '_snapshot_binary_sum'],
    )
    group_count = looked_up['_snapshot_binary_count'].fillna(0)
    group_sum = looked_up['_snapshot_binary_sum'].fillna(0)
    return pd.DataFrame({
        f'past_closure_count_{group_col}': group_count.astype(int),
        f'past_closure_rate_{group_col}': (
            group_sum + HISTORY_SMOOTHING_ALPHA * global_rate
        ) / (group_count + HISTORY_SMOOTHING_ALPHA),
    }, index=data.index)

def build_past_duration_features(data, group_col, global_stats):
    keys = history_key(data[group_col])
    snapshots = make_duration_snapshots(data, keys, 'duration_label_available_at')
    snapshot_cols = [
        '_snapshot_duration_count', '_snapshot_duration_q25',
        '_snapshot_duration_median', '_snapshot_duration_q75',
        *[f'_snapshot_duration_rate_{name}' for name in DURATION_CLASSES],
    ]
    looked_up = strict_asof_lookup(
        data, 'prediction_datetime', keys, snapshots, 'duration_label_available_at', snapshot_cols
    )
    group_count = looked_up['_snapshot_duration_count'].fillna(0)
    feature_data = {f'past_duration_count_{group_col}': group_count.astype(int)}

    enough_history = group_count.ge(MIN_DURATION_GROUP_HISTORY)
    for statistic in ['q25', 'median', 'q75']:
        group_value = looked_up[f'_snapshot_duration_{statistic}']
        feature_data[f'past_duration_{statistic}_{group_col}'] = group_value.where(
            enough_history, global_stats[f'past_duration_global_{statistic}']
        )

    for duration_class in DURATION_CLASSES:
        group_rate = looked_up[f'_snapshot_duration_rate_{duration_class}'].fillna(0)
        global_rate = global_stats[f'past_duration_global_rate_{duration_class}']
        feature_data[f'past_duration_rate_{duration_class}_{group_col}'] = (
            group_rate * group_count + HISTORY_SMOOTHING_ALPHA * global_rate
        ) / (group_count + HISTORY_SMOOTHING_ALPHA)

    return pd.DataFrame(feature_data, index=data.index)

In [84]:
# Make reruns idempotent and defragment the source frame before heavy history work.
existing_history_cols = [col for col in df.columns if col.startswith('past_')]
history_source_df = df.drop(columns=existing_history_cols, errors='ignore').copy()

history_group_cols = [
    col for col in [
        'event_cause', 'corridor', 'zone', 'junction', 'police_station',
        'location_grid', 'event_cause_corridor',
    ] if col in history_source_df.columns
]
history_feature_parts = []

for col in history_group_cols:
    history_feature_parts.append(build_strict_past_event_count(history_source_df, col))

global_keys = pd.Series(GLOBAL_HISTORY_KEY, index=history_source_df.index, dtype='string')
global_closure_snapshots = make_binary_snapshots(
    history_source_df, global_keys, 'target_road_closure', 'prediction_datetime'
)
global_closure_lookup = strict_asof_lookup(
    history_source_df, 'prediction_datetime', global_keys,
    global_closure_snapshots, 'prediction_datetime',
    ['_snapshot_binary_count', '_snapshot_binary_sum'],
)
global_closure_features = pd.DataFrame({
    'past_closure_global_count': (
        global_closure_lookup['_snapshot_binary_count'].fillna(0).astype(int)
    ),
    'past_closure_global_rate': (
        global_closure_lookup['_snapshot_binary_sum']
        / global_closure_lookup['_snapshot_binary_count']
    ).fillna(0.5),
}, index=history_source_df.index)
history_feature_parts.append(global_closure_features)

for col in history_group_cols:
    history_feature_parts.append(build_past_closure_features(
        history_source_df, col, global_closure_features['past_closure_global_rate']
    ))

global_duration_snapshots = make_duration_snapshots(
    history_source_df, global_keys, 'duration_label_available_at'
)
global_duration_snapshot_cols = [
    '_snapshot_duration_count', '_snapshot_duration_q25',
    '_snapshot_duration_median', '_snapshot_duration_q75',
    *[f'_snapshot_duration_rate_{name}' for name in DURATION_CLASSES],
]
global_duration_lookup = strict_asof_lookup(
    history_source_df, 'prediction_datetime', global_keys, global_duration_snapshots,
    'duration_label_available_at', global_duration_snapshot_cols,
)
global_duration_feature_data = {
    'past_duration_global_count': (
        global_duration_lookup['_snapshot_duration_count'].fillna(0).astype(int)
    ),
}
for statistic in ['q25', 'median', 'q75']:
    global_duration_feature_data[f'past_duration_global_{statistic}'] = (
        global_duration_lookup[f'_snapshot_duration_{statistic}']
    )
for duration_class in DURATION_CLASSES:
    global_duration_feature_data[f'past_duration_global_rate_{duration_class}'] = (
        global_duration_lookup[f'_snapshot_duration_rate_{duration_class}'].fillna(0.25)
    )
global_duration_features = pd.DataFrame(
    global_duration_feature_data, index=history_source_df.index
)
history_feature_parts.append(global_duration_features)

global_duration_stats = global_duration_features[[
    'past_duration_global_q25', 'past_duration_global_median', 'past_duration_global_q75',
    *[f'past_duration_global_rate_{name}' for name in DURATION_CLASSES],
]]
for col in history_group_cols:
    history_feature_parts.append(build_past_duration_features(
        history_source_df, col, global_duration_stats
    ))

# Add every historical column in one operation to avoid DataFrame fragmentation.
history_feature_frame = pd.concat(history_feature_parts, axis=1)
assert not history_feature_frame.columns.duplicated().any(), (
    'Duplicate historical feature names were generated.'
)
df = pd.concat([history_source_df, history_feature_frame], axis=1)
history_feature_cols = history_feature_frame.columns.tolist()
print('Historical features created:', len(history_feature_cols))

Historical features created: 87


## 9. Historical-feature safeguards

The assertions below verify non-negative counts and identical history for events sharing a group and prediction timestamp. The helper implementation also uses strict as-of matching, which prevents current/same-time labels from entering their own features.

In [85]:
history_count_cols = [
    col for col in history_feature_cols
    if col.startswith('past_count_') or '_count_' in col or col.endswith('_count')
]
for col in history_count_cols:
    assert df[col].dropna().ge(0).all(), f'Negative historical count found in {col}'

same_time_history_issues = []
for group_col in history_group_cols:
    check_cols = [
        f'past_count_{group_col}',
        f'past_closure_count_{group_col}',
        f'past_closure_rate_{group_col}',
        f'past_duration_count_{group_col}',
    ]
    check_cols = [col for col in check_cols if col in df.columns]
    if not check_cols:
        continue
    grouped_nunique = df.groupby(
        [group_col, 'prediction_datetime'], dropna=False
    )[check_cols].nunique(dropna=False)
    for col in check_cols:
        if grouped_nunique[col].gt(1).any():
            same_time_history_issues.append(f'{group_col}:{col}')

assert not same_time_history_issues, (
    'Same-time events received different prior history: '
    f'{same_time_history_issues[:20]}'
)
print('Historical count and same-timestamp checks passed.')

Historical count and same-timestamp checks passed.


## 10. Define separate feature sets and block leakage

The two tasks share safe base features, but duration receives additional past-duration features. The latest ensemble and duration-model importances are used conservatively: only direct features or complete one-hot feature families with exactly zero importance in both latest models are excluded. Low-but-nonzero, model-specific-zero, and newly created v1 features are retained for later validation. No global imputation, rare-category fitting, or one-hot encoding is performed here.

In [86]:
blocked_features = {
    # Targets and their raw/derived sources.
    'requires_road_closure', 'target_road_closure', 'target_road_closure_missing',
    'duration_minutes_raw', 'duration_band', 'duration_source',
    'duration_label_available_at', 'has_duration_label',
    'end_datetime', 'closed_datetime', 'resolved_datetime',
    # Post-event state and ownership.
    'status', 'modified_datetime', 'closed_by_id', 'resolved_by_id',
    'resolved_at_address', 'resolved_at_latitude', 'resolved_at_longitude',
    'last_modified_by_id', 'comment',
    # Endpoint/route fields and known closure proxies.
    'endlatitude', 'endlongitude', 'end_address', 'route_path', 'map_file',
    'has_end_location', 'end_point_missing_or_zero', 'route_distance_km', 'has_route_span',
    # QA-only duration alternatives and flags.
    'qa_duration_minutes_closed', 'qa_duration_minutes_end', 'qa_duration_minutes_resolved',
    'qa_negative_duration_closed', 'qa_negative_duration_end', 'qa_negative_duration_resolved',
    'qa_duration_valid_source_count', 'qa_duration_source_spread_minutes',
    'qa_any_negative_duration',
}

# These features had exactly zero importance in both latest model runs.
# Categorical evidence is the sum across every previous one-hot level in that family.
previous_zero_importance_features = {
    'valid_start_coordinate',
    'has_start_location',
    'report_lag_is_negative',
    'has_emergency_word',
    'is_two_wheeler',
    'has_cargo_material',
    'truck_age_missing',
    'truck_age_band',
    'cargo_material',
    'direction',
    'reason_breakdown',
}

previous_importance_exclusion_audit = pd.DataFrame([
    {'feature': 'valid_start_coordinate', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'has_start_location', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'report_lag_is_negative', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'has_emergency_word', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'is_two_wheeler', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'has_cargo_material', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'truck_age_missing', 'feature_kind': 'direct', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'truck_age_band', 'feature_kind': 'one_hot_family_sum', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'cargo_material', 'feature_kind': 'one_hot_family_sum', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'direction', 'feature_kind': 'one_hot_family_sum', 'model1_importance': 0.0, 'model2_importance': 0.0},
    {'feature': 'reason_breakdown', 'feature_kind': 'one_hot_family_sum', 'model1_importance': 0.0, 'model2_importance': 0.0},
])
previous_importance_exclusion_audit['decision'] = 'exclude_from_both_v1_feature_sets'
previous_importance_exclusion_audit['evidence_rule'] = (
    'Exactly zero importance in both latest models; categorical values aggregated across all dummy levels.'
)
previous_importance_exclusion_audit['model1_source'] = 'feature_importance_ensemble.csv'
previous_importance_exclusion_audit['model2_source'] = 'model2_v1_duration_band_feature_importance.csv'

# Automated data-quality exclusions from the current v1 feature tables.
quality_excluded_features_both = {'age_of_truck'}
duration_only_quality_excluded_features = {'report_lag_missing'}

base_numeric_candidates = [
    'latitude', 'longitude', 'valid_start_coordinate', 'has_start_location',
    'lat_round_3', 'lon_round_3', 'distance_to_city_center_km',
    'start_hour', 'start_dayofweek', 'start_month_number', 'start_weekofyear',
    'is_weekend', 'is_morning_peak', 'is_evening_peak', 'is_peak_hour', 'is_night',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'report_lag_minutes_clipped', 'report_lag_hours_clipped',
    'report_lag_missing', 'report_lag_is_negative',
    'description_missing', 'text_length', 'description_char_length',
    'description_word_count', 'has_non_ascii_text', 'has_kannada_text',
    'has_accident_word', 'has_breakdown_word', 'has_water_word',
    'has_construction_word', 'has_event_word', 'has_blocked_word',
    'has_jam_word', 'has_vip_word', 'has_location_hint_word', 'has_emergency_word',
    'is_planned_event', 'is_public_or_vip_event', 'is_breakdown_event',
    'is_accident_event', 'is_weather_or_visibility_event', 'is_road_condition_event',
    'has_vehicle_type', 'is_truck', 'is_bus', 'is_two_wheeler', 'is_heavy_vehicle',
    'has_cargo_material', 'age_of_truck', 'truck_age_missing',
]
base_categorical_candidates = [
    'event_type', 'event_cause', 'direction', 'veh_type', 'corridor',
    'cargo_material', 'reason_breakdown', 'police_station', 'zone', 'junction',
    'location_grid', 'truck_age_band', 'peak_period', 'start_day_name', 'start_month_name',
    'event_cause_corridor', 'cause_peak_interaction', 'zone_cause_interaction',
    'corridor_cause_interaction', 'cause_heavy_vehicle_interaction',
    'corridor_peak_interaction', 'cause_hour_interaction',
    'vehicle_cause_interaction', 'cargo_vehicle_interaction',
    'planned_peak_interaction', 'weather_zone_interaction',
]

base_numeric_features = [
    col for col in base_numeric_candidates
    if (
        col in df.columns
        and col not in previous_zero_importance_features
        and col not in quality_excluded_features_both
    )
]
base_categorical_features = [
    col for col in base_categorical_candidates
    if col in df.columns and col not in previous_zero_importance_features
]
past_event_features = [col for col in df.columns if col.startswith('past_count_')]
past_closure_features = [col for col in df.columns if col.startswith('past_closure_')]
past_duration_features = [col for col in df.columns if col.startswith('past_duration_')]

road_closure_feature_cols = list(dict.fromkeys(
    base_numeric_features + base_categorical_features
    + past_event_features + past_closure_features
))
duration_feature_cols = [
    col for col in list(dict.fromkeys(
        base_numeric_features + base_categorical_features
        + past_event_features + past_closure_features + past_duration_features
    ))
    if col not in duration_only_quality_excluded_features
]

feature_cols = road_closure_feature_cols
leaked_features = set(blocked_features) & set(feature_cols)

assert not leaked_features, (
    f'Blocked features found in training data: {sorted(leaked_features)}'
)
assert not previous_zero_importance_features.intersection(feature_cols), (
    'Previously zero-importance features found in road-closure features: '
    f'{sorted(previous_zero_importance_features.intersection(feature_cols))}'
)
assert not quality_excluded_features_both.intersection(feature_cols), (
    'Data-quality exclusions found in road-closure features: '
    f'{sorted(quality_excluded_features_both.intersection(feature_cols))}'
)

feature_cols = duration_feature_cols
leaked_features = set(blocked_features) & set(feature_cols)

assert not leaked_features, (
    f'Blocked features found in training data: {sorted(leaked_features)}'
)
assert not previous_zero_importance_features.intersection(feature_cols), (
    'Previously zero-importance features found in duration features: '
    f'{sorted(previous_zero_importance_features.intersection(feature_cols))}'
)
assert not quality_excluded_features_both.intersection(feature_cols), (
    'Shared data-quality exclusions found in duration features: '
    f'{sorted(quality_excluded_features_both.intersection(feature_cols))}'
)
assert not duration_only_quality_excluded_features.intersection(feature_cols), (
    'Duration-only data-quality exclusions found in duration features: '
    f'{sorted(duration_only_quality_excluded_features.intersection(feature_cols))}'
)

assert len(road_closure_feature_cols) == len(set(road_closure_feature_cols))
assert len(duration_feature_cols) == len(set(duration_feature_cols))
print('Road-closure features:', len(road_closure_feature_cols))
print('Duration features:', len(duration_feature_cols))

Road-closure features: 91
Duration features: 154


## 11. Build clean feature tables

Missing values and categorical values are preserved. The later modelling pipeline must fit imputers, rare-category rules, and encoders on its training split only.

In [87]:
metadata_cols = [
    col for col in ['id', '_source_row', 'prediction_datetime', 'start_datetime']
    if col in df.columns
]

road_closure_mask = df['target_road_closure'].notna()
road_closure_feature_table = df.loc[
    road_closure_mask, metadata_cols + road_closure_feature_cols + ['target_road_closure']
].copy()
road_closure_feature_table['target_road_closure'] = (
    road_closure_feature_table['target_road_closure'].astype(int)
)

duration_mask = df['duration_minutes_raw'].notna() & df['duration_band'].notna()
duration_feature_table = df.loc[
    duration_mask, metadata_cols + duration_feature_cols
    + ['duration_minutes_raw', 'duration_band']
].copy()
duration_feature_table['duration_band'] = duration_feature_table['duration_band'].astype('string')

assert not set(blocked_features).intersection(road_closure_feature_cols)
assert not set(blocked_features).intersection(duration_feature_cols)
assert not set(col for col in duration_feature_table.columns if col.startswith('qa_duration_'))

print('Road-closure table:', road_closure_feature_table.shape)
print('Duration table:', duration_feature_table.shape)

Road-closure table: (8173, 96)
Duration table: (3498, 160)


## 12. Feature manifest and quality report

Every raw or engineered column receives a status. Columns that were merely ignored are no longer silently lost or incorrectly labelled as leakage.

In [88]:
target_columns = {'target_road_closure', 'duration_minutes_raw', 'duration_band'}
qa_columns = {col for col in df.columns if col.startswith('qa_')} | {
    'duration_source', 'duration_label_available_at', 'has_duration_label',
    'target_road_closure_missing',
}
identifier_columns = {
    'id', 'veh_no', 'client_id', 'created_by_id', 'assigned_to_police_id',
    'citizen_accident_id', 'kgid', 'gba_identifier',
}
replaced_raw_columns = {
    'description', 'address', 'created_date', 'start_datetime',
    'latitude', 'longitude', 'age_of_truck',
}

def model_column_status(col, selected_features):
    if col in selected_features:
        return 'included_feature'
    if col in target_columns:
        return 'target_only'
    if col in metadata_cols:
        return 'metadata_only'
    if col in previous_zero_importance_features:
        return 'excluded_previous_zero_importance'
    if col in quality_excluded_features_both or col in duration_only_quality_excluded_features:
        return 'excluded_data_quality'
    if col in blocked_features:
        return 'blocked_leakage_or_target_source'
    if col in qa_columns:
        return 'qa_only'
    if col in identifier_columns:
        return 'excluded_identifier'
    if col in replaced_raw_columns:
        return 'replaced_by_engineered_feature'
    return 'excluded_not_selected_or_needs_review'

def exclusion_reason(col):
    if col in target_columns:
        return 'Supervised target; never part of X.'
    if col in metadata_cols:
        return 'Retained only for identity, ordering, split, and audit.'
    if col in previous_zero_importance_features:
        return 'Exactly zero importance in both latest model runs; categorical families were aggregated across all dummy levels.'
    if col == 'age_of_truck':
        return 'Excluded because it is more than 96% missing in both current training tables.'
    if col == 'report_lag_missing':
        return 'Excluded from duration because it is constant in the duration training subset.'
    if col in blocked_features:
        return 'Known post-event field, target source, endpoint proxy, or QA-only derivative.'
    if col in qa_columns:
        return 'Retained only for target/data-quality QA.'
    if col in identifier_columns:
        return 'Identifier/high-cardinality field; not treated as leakage.'
    if col in replaced_raw_columns:
        return 'Raw source replaced by a prediction-safe engineered representation.'
    if col in {'priority', 'priority_clean', 'authenticated', 'authenticated_clean'}:
        return 'Availability at first report is not yet verified.'
    return 'Not selected; document availability and expected utility before adding.'

manifest_rows = []
for col in sorted(df.columns):
    if col in road_closure_feature_cols or col in duration_feature_cols:
        if pd.api.types.is_numeric_dtype(df[col]):
            data_type = 'numeric'
        else:
            data_type = 'categorical'
    elif col in target_columns:
        data_type = 'target'
    elif col in metadata_cols:
        data_type = 'metadata'
    elif col in qa_columns:
        data_type = 'qa'
    else:
        data_type = str(df[col].dtype)

    manifest_rows.append({
        'column': col,
        'data_type': data_type,
        'road_closure_status': model_column_status(col, road_closure_feature_cols),
        'road_closure_reason': (
            '' if col in road_closure_feature_cols else exclusion_reason(col)
        ),
        'duration_status': model_column_status(col, duration_feature_cols),
        'duration_reason': (
            '' if col in duration_feature_cols else exclusion_reason(col)
        ),
    })

feature_manifest = pd.DataFrame(manifest_rows)

quality_rows = []
for table_name, table, selected_features in [
    ('road_closure', road_closure_feature_table, road_closure_feature_cols),
    ('duration', duration_feature_table, duration_feature_cols),
]:
    for col in selected_features:
        numeric = pd.to_numeric(table[col], errors='coerce') if pd.api.types.is_numeric_dtype(table[col]) else None
        infinity_count = int(np.isinf(numeric).sum()) if numeric is not None else 0
        quality_rows.append({
            'table': table_name,
            'feature': col,
            'dtype': str(table[col].dtype),
            'rows': len(table),
            'missing_count': int(table[col].isna().sum()),
            'missing_percent': float(table[col].isna().mean() * 100),
            'unique_values': int(table[col].nunique(dropna=True)),
            'infinite_count': infinity_count,
        })

feature_quality_report = pd.DataFrame(quality_rows)

# Consolidated automatic QA: FAIL stops the notebook; WARN gives a targeted action.
qa_check_rows = []

def record_qa(check, status, observed, rule, action):
    qa_check_rows.append({
        'check': check, 'status': status, 'observed': str(observed),
        'rule': rule, 'action': action,
    })

blocked_in_output = sorted(
    set(blocked_features).intersection(road_closure_feature_cols)
    | set(blocked_features).intersection(duration_feature_cols)
)
record_qa(
    'blocked leakage features absent', 'PASS' if not blocked_in_output else 'FAIL',
    blocked_in_output or 'none', 'No blocked field may enter either feature set.',
    'Remove every listed field from feature selection.',
)

zero_importance_in_output = sorted(
    previous_zero_importance_features.intersection(road_closure_feature_cols)
    | previous_zero_importance_features.intersection(duration_feature_cols)
)
record_qa(
    'latest zero-importance exclusions applied',
    'PASS' if not zero_importance_in_output else 'FAIL',
    zero_importance_in_output or 'none',
    'Latest agreed zero-importance fields must be absent.',
    'Apply the configured importance exclusion set.',
)

selected_negative_duration_count = int(df['duration_minutes_raw'].lt(0).sum())
record_qa(
    'negative selected durations removed',
    'PASS' if selected_negative_duration_count == 0 else 'FAIL',
    selected_negative_duration_count, 'Selected duration must be >= 0 minutes.',
    'Invalidate negative source durations before applying the hierarchy.',
)

expected_duration_source = pd.Series(pd.NA, index=df.index, dtype='string')
closed_valid = valid_duration_by_source['closed'].notna()
end_valid = valid_duration_by_source['end'].notna()
resolved_valid = valid_duration_by_source['resolved'].notna()
expected_duration_source.loc[closed_valid] = 'closed'
expected_duration_source.loc[~closed_valid & end_valid] = 'end'
expected_duration_source.loc[~closed_valid & ~end_valid & resolved_valid] = 'resolved'
hierarchy_violations = int((
    df['duration_source'].fillna('__missing__')
    != expected_duration_source.fillna('__missing__')
).sum())
record_qa(
    'duration hierarchy respected', 'PASS' if hierarchy_violations == 0 else 'FAIL',
    hierarchy_violations, 'Use closed, otherwise end, otherwise resolved.',
    'Correct duration-source selection logic.',
)

valid_duration_mask = df['duration_minutes_raw'].notna()
very_long_expected = df['duration_minutes_raw'].gt(24 * 60)
very_long_actual = df['duration_band'].astype('string').eq('very_long')
very_long_mismatches = int((
    valid_duration_mask & very_long_expected.ne(very_long_actual)
).sum())
record_qa(
    'very-long duration class correct',
    'PASS' if very_long_mismatches == 0 else 'FAIL', very_long_mismatches,
    'Every duration > 1440 minutes must be very_long, and no shorter row may be very_long.',
    'Correct duration bin definitions.',
)

boundary_warning_count = (
    int(duration_boundary_summary['status'].eq('WARN').sum())
    if 'status' in duration_boundary_summary.columns else 0
)
record_qa(
    'duration boundary rounding concentration',
    'WARN' if boundary_warning_count else 'PASS', boundary_warning_count,
    'Warn when >=10 exact-boundary rows form >=50% of nearby rows and >=90% are minute-aligned.',
    'If warned, inspect only flagged rows in duration_boundary_qa; otherwise no review is needed.',
)

record_qa(
    'same-time history isolation', 'PASS' if not same_time_history_issues else 'FAIL',
    same_time_history_issues[:10] or 'none',
    'Rows sharing group and prediction time must receive identical prior history.',
    'Use strict as-of matching with exact matches disabled.',
)

history_rate_cols = [
    col for col in history_feature_cols if '_rate' in col
]
history_rate_violations = int(sum(
    ((df[col] < 0) | (df[col] > 1)).fillna(False).sum()
    for col in history_rate_cols
))
record_qa(
    'historical rates bounded', 'PASS' if history_rate_violations == 0 else 'FAIL',
    history_rate_violations, 'Every historical rate must be between 0 and 1.',
    'Correct rate smoothing or cumulative-count logic.',
)

infinite_feature_values = int(feature_quality_report['infinite_count'].sum())
record_qa(
    'selected features contain no infinity',
    'PASS' if infinite_feature_values == 0 else 'FAIL', infinite_feature_values,
    'Selected features must not contain +/- infinity.',
    'Replace invalid divisions with missing values before saving.',
)

high_missing_features = feature_quality_report.loc[
    feature_quality_report['missing_percent'].ge(HIGH_MISSING_PERCENT_WARN),
    ['table', 'feature', 'missing_percent'],
].to_dict('records')
record_qa(
    'extreme feature missingness', 'WARN' if high_missing_features else 'PASS',
    high_missing_features[:10] or 'none',
    f'Warn when a selected feature is >= {HIGH_MISSING_PERCENT_WARN:.0f}% missing.',
    'If warned, consider excluding the listed feature or adding a justified missingness strategy.',
)

constant_features = feature_quality_report.loc[
    feature_quality_report['unique_values'].le(1), ['table', 'feature']
].to_dict('records')
record_qa(
    'constant selected features', 'WARN' if constant_features else 'PASS',
    constant_features[:20] or 'none', 'Selected features should vary across rows.',
    'If warned, remove listed constants before modelling.',
)

duplicate_id_count = (
    int(df['id'].duplicated().sum()) if 'id' in df.columns else 0
)
record_qa(
    'duplicate event identifiers', 'WARN' if duplicate_id_count else 'PASS',
    duplicate_id_count, 'Event identifiers should normally be unique.',
    'If warned, verify whether rows are updates or true duplicates before modelling.',
)

missing_duration_classes = sorted(
    set(DURATION_CLASSES)
    - set(duration_feature_table['duration_band'].dropna().astype(str).unique())
)
record_qa(
    'all configured duration classes represented',
    'WARN' if missing_duration_classes else 'PASS',
    missing_duration_classes or 'all present',
    'Every configured class should have at least one training row.',
    'If warned, reconsider the class or obtain examples before model training.',
)

chronological_tables = (
    road_closure_feature_table['prediction_datetime'].dropna().is_monotonic_increasing
    and duration_feature_table['prediction_datetime'].dropna().is_monotonic_increasing
)
record_qa(
    'feature tables remain chronological', 'PASS' if chronological_tables else 'FAIL',
    chronological_tables, 'Both outputs must remain sorted by prediction time.',
    'Sort before constructing and saving feature tables.',
)

qa_check_summary = pd.DataFrame(qa_check_rows)
print(qa_check_summary.to_string(index=False))
qa_failures = qa_check_summary.loc[qa_check_summary['status'].eq('FAIL')]
qa_warnings = qa_check_summary.loc[qa_check_summary['status'].eq('WARN')]
if not qa_warnings.empty:
    print('\nQA warnings require targeted review:')
    print(qa_warnings[['check', 'observed', 'action']].to_string(index=False))
assert qa_failures.empty, (
    'Feature engineering QA failed:\n'
    + qa_failures[['check', 'observed', 'action']].to_string(index=False)
)

qa_check_summary

                                      check status    observed                                                                                       rule                                                                                       action
            blocked leakage features absent   PASS        none                                             No blocked field may enter either feature set.                                            Remove every listed field from feature selection.
  latest zero-importance exclusions applied   PASS        none                                       Latest agreed zero-importance fields must be absent.                                               Apply the configured importance exclusion set.
        negative selected durations removed   PASS           0                                                    Selected duration must be >= 0 minutes.                          Invalidate negative source durations before applying the hierarchy.
            

,check,status,observed,rule,action
0,blocked leakage features absent,PASS,none,No blocked field may enter either feature set.,Remove every listed field from feature selection.
1,latest zero-importance exclusions applied,PASS,none,Latest agreed zero-importance fields must be a...,Apply the configured importance exclusion set.
2,negative selected durations removed,PASS,0,Selected duration must be >= 0 minutes.,Invalidate negative source durations before ap...
3,duration hierarchy respected,PASS,0,"Use closed, otherwise end, otherwise resolved.",Correct duration-source selection logic.
4,very-long duration class correct,PASS,0,Every duration > 1440 minutes must be very_lon...,Correct duration bin definitions.
5,duration boundary rounding concentration,PASS,0,Warn when >=10 exact-boundary rows form >=50% ...,"If warned, inspect only flagged rows in durati..."
6,same-time history isolation,PASS,none,Rows sharing group and prediction time must re...,Use strict as-of matching with exact matches d...
7,historical rates bounded,PASS,0,Every historical rate must be between 0 and 1.,Correct rate smoothing or cumulative-count logic.
8,selected features contain no infinity,PASS,0,Selected features must not contain +/- infinity.,Replace invalid divisions with missing values ...
9,extreme feature missingness,PASS,none,Warn when a selected feature is >= 80% missing.,"If warned, consider excluding the listed featu..."


## 13. Save the two model input tables

Only the road-closure training table and duration base training table are written to CSV. The training tables remain unencoded and unimputed. Manifests, duration alternatives, invalid-negative summaries, boundary diagnostics, and quality reports remain available inside the notebook without creating additional files.

In [89]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([
    {'metric': 'raw_rows', 'value': len(raw_df)},
    {'metric': 'road_closure_training_rows', 'value': len(road_closure_feature_table)},
    {'metric': 'duration_training_rows', 'value': len(duration_feature_table)},
    {'metric': 'road_closure_features', 'value': len(road_closure_feature_cols)},
    {'metric': 'duration_features', 'value': len(duration_feature_cols)},
    {'metric': 'latest_zero_importance_features_excluded', 'value': len(previous_zero_importance_features)},
    {'metric': 'missing_road_closure_targets_removed', 'value': int(df['target_road_closure'].isna().sum())},
    {'metric': 'rows_with_any_negative_duration', 'value': int(df['qa_any_negative_duration'].sum())},
    {'metric': 'valid_duration_targets', 'value': int(df['duration_minutes_raw'].notna().sum())},
    *[
        {'metric': f'duration_class_{name}', 'value': int(df['duration_band'].astype('string').eq(name).sum())}
        for name in DURATION_CLASSES
    ],
])

outputs = {
    'road_closure_features_v1.csv': road_closure_feature_table,
    'duration_base_features_v1.csv': duration_feature_table,
}

saved_files = []
for file_name, frame in outputs.items():
    output_path = OUTPUT_DIR / file_name
    frame.to_csv(output_path, index=False)
    saved_files.append(output_path)

saved_files

[WindowsPath('D:/Python/Gridlock/Phase 2/theme 2/outputs/feature_engineering_v1/road_closure_features_v1.csv'),
 WindowsPath('D:/Python/Gridlock/Phase 2/theme 2/outputs/feature_engineering_v1/duration_base_features_v1.csv')]

## 14. Final feature-engineering summary

In [90]:
print(summary.to_string(index=False))
print('\nDuration hierarchy: closed -> end -> resolved')
print('Duration classes: short <= 60, medium <= 240, long <= 1440, very_long > 1440 minutes')
print('Global imputation performed: no')
print('Global rare-category fitting performed: no')
print('Global one-hot encoding performed: no')
print('Output directory:', OUTPUT_DIR)

                                  metric  value
                                raw_rows   8173
              road_closure_training_rows   8173
                  duration_training_rows   3498
                   road_closure_features     91
                       duration_features    154
latest_zero_importance_features_excluded     11
    missing_road_closure_targets_removed      0
         rows_with_any_negative_duration     51
                  valid_duration_targets   3498
                    duration_class_short   1579
                   duration_class_medium    903
                     duration_class_long    280
                duration_class_very_long    736

Duration hierarchy: closed -> end -> resolved
Duration classes: short <= 60, medium <= 240, long <= 1440, very_long > 1440 minutes
Global imputation performed: no
Global rare-category fitting performed: no
Global one-hot encoding performed: no
Output directory: D:\Python\Gridlock\Phase 2\theme 2\outputs\feature_engineering_v1